### NYC DOT Marketing Mix Modeling -- Feature Engineering

#### Import libraries

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np
from datetime import date, timedelta
from pathlib import Path

# Show all columns
pd.set_option('display.max_columns', None)

#### File paths

In [2]:
# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import data

In [3]:
data = pd.read_parquet(CLEAN_DATA_DIR / "01_dot_mmm_preprocessing.parquet")
data.head()

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital_social,spend_safety_ooh_other,crashes,injured,killed,t,is_holiday_week,post_congestion
0,2022-07-04,0.00,0.00,0.00,0.00,0.00,1919,1053,12,0,1,0
1,2022-07-11,21920.83,3129.63,3209.92,0.00,18135.30,1994,1191,9,1,0,0
2,2022-07-18,25574.30,3651.24,3744.90,0.00,21157.85,2086,1183,10,2,0,0
3,2022-07-25,25574.30,3651.24,53479.90,5730.00,35958.85,2044,1020,5,3,0,0
4,2022-08-01,25574.30,3651.24,60952.90,2996.45,36106.01,2113,1127,5,4,0,0


In [4]:
# Number of rows, columns
data.shape

(156, 12)

#### Data types and nulls

In [5]:
# Data types and nulls
data_types_null_counts = pd.DataFrame()
data_types_null_counts["Data Type"] = data.dtypes
data_types_null_counts["Count of Nulls"] = data.isna().sum()
data_types_null_counts

,Data Type,Count of Nulls
week_start,datetime64[ns],0
spend_safety_tv,float64,0
spend_safety_radio,float64,0
spend_safety_print,float64,0
spend_safety_digital_social,float64,0
spend_safety_ooh_other,float64,0
crashes,int64,0
injured,int64,0
killed,int64,0
t,int64,0


#### Seasonal Features
Created sine and cosine seasonal features to represent the cyclical nature of the calendar year, allowing the model to recognize that the end and beginning of the year are adjacent rather than numerically distant.

In [6]:
# Add columns
day_of_year = data["week_start"].dt.dayofyear
data["season_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
data["season_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)

#### Time Trend Scaling

Applied MinMaxScaler to normalize the sequential time variable to a 0–1 range, reducing its numerical magnitude while preserving the underlying temporal trend. Added the scaled variable (t_scaled) to the dataset for modeling.

In [7]:
# Import scaler
from sklearn.preprocessing import MinMaxScaler

In [8]:
# Initialize the scaler
scaler = MinMaxScaler()

# Fit and transform
t_scaled = scaler.fit_transform(data[["t"]])

In [9]:
# Add 't_scaled'
data["t_scaled"] = t_scaled

#### Final Data Validation

Final validation checks to ensure data integrity was maintained, all required features and the target variable were present, and no missing values, duplicate weeks, negative spending values, or gaps in weekly observations had been introduced.

In [10]:
# Spending columns
spend_cols = [
    "spend_safety_tv", 
    "spend_safety_radio",
    "spend_safety_print",
    "spend_safety_digital_social", 
    "spend_safety_ooh_other"
]

In [11]:
# Final validation
print("Shape:", data.shape)
print("Missing values:", data.isna().sum().sum())
print("Duplicate dates:", data["week_start"].duplicated().sum())
print("Negative spending:", (data[spend_cols] < 0).sum().sum())

# Verify weekly continuity
print("Weekly gaps:", (
    data["week_start"].diff().dropna() != pd.Timedelta(days=7)
).sum())

Shape: (156, 15)
Missing values: 0
Duplicate dates: 0
Negative spending: 0
Weekly gaps: 0


#### Export data

In [12]:
data.to_parquet(
    CLEAN_DATA_DIR / "03_dot_mmm_feature_engineering.parquet",
    index=False
)